# core

> Minimal IPython notebook integration for an LLM `%%prompt` cell magic. Inspired by fast.ai's solveit.

The whole module is built around three observations:

1. An `.ipynb` is just JSON. Cells have sources, outputs, and (since nbformat 4.5) stable ids.
2. JupyterLab tells the kernel which cell is executing via `parent_header.metadata.cellId`. That's enough to locate ourselves in the notebook without text matching.
3. Cell outputs persist in the JSON on disk. If a prompt cell already has output, re-running it shouldn't cost tokens.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from IPython.display import Markdown, display
from IPython.core.magic import register_cell_magic

from pathlib import Path
import argparse, json, os, shlex

## System prompt

Kept short and honest about the actual environment. The model takes its tone, structure, and capabilities cues from here, so lies about features we don't have (tool use, variable injection, etc.) leak into responses.

In [ ]:
#| export
system_prompt = """You are an AI assistant embedded inside an IPython notebook through a `%%prompt` cell magic. The notebook is a linear sequence of markdown, code, and prompt cells executed in a single persistent kernel — state from code cells carries forward. When the user runs a prompt cell, every cell above it (sources and captured outputs) is sent to you as the conversation history; previous prompt cells appear as your prior assistant turns. The dialog *is* the notebook.

Be concise, direct, and incremental. Match response length to the question. Do not pad, restate, or end with "let me know if...". Use fenced code blocks with language tags. Default to idiomatic Python — comprehensions, broadcasting, fastcore-style brevity. Short single-line docstrings; no inline comments unless a constraint is genuinely non-obvious.

Your knowledge cutoff is January 2026."""

## Locating the current notebook

Two-step strategy. JupyterLab ≥ 3.5 sets `JPY_SESSION_NAME` to the notebook's path; use it to determine the current notebook's path.

In [ ]:
#| export
def _current_notebook():
    "Resolve the path of the notebook this kernel is serving."
    name = os.environ.get("JPY_SESSION_NAME")
    if name and Path(name).exists(): return Path(name)
    raise RuntimeError("Cannot determine current notebook; save the file or set JPY_SESSION_NAME")

In [ ]:
#| eval: false
_current_notebook()

Path('/Users/pablo/src/nbdialog/nbs/00_core.ipynb')

## Cells → chat messages

Prompt cells contribute one `user` message (the prompt body with the `%%prompt` line stripped) and, when they already have rendered output, one `assistant` message replaying that output. Code and markdown cells contribute their source as `user`, plus any captured outputs as a follow-up `user` message. The cell whose id matches `up_to_id` is the boundary — included for its prompt but never for its (stale) cached response, since that response is what the current call is producing.

In [ ]:
#| export
def _join(x): return "".join(x) if isinstance(x, list) else x

In [ ]:
#| export
def _is_prompt(cell):
    "True if `cell` is a code cell whose source starts with the %%prompt magic."
    if cell.get("cell_type") != "code": return False
    return _join(cell.get("source", "")).lstrip().startswith("%%prompt")

In [ ]:
#| export
def _strip_magic(src):
    "Drop the leading %%prompt line from a prompt cell's source."
    return "\n".join(_join(src).splitlines()[1:])

In [ ]:
#| export
def _output_text(out):
    "Extract a readable string from one nbformat output entry (for context, not response)."
    if out.get("output_type") == "stream": return _join(out.get("text", []))
    data = out.get("data", {})
    if "text/markdown" in data: return _join(data["text/markdown"])
    if "text/plain"    in data: return _join(data["text/plain"])
    return ""

def _response_markdown(cell):
    "Return the model's rendered markdown response stored in this cell's outputs, or None."
    for o in cell.get("outputs", []):
        data = o.get("data", {})
        if "text/markdown" in data: return _join(data["text/markdown"])
    return None

In [ ]:
#| export
def notebook_to_messages(cells, up_to_id, system=None):
    "Build chat messages from `cells` up to and including the cell with id `up_to_id`."
    msgs = [{"role": "system", "content": system or system_prompt}]
    for c in cells:
        src = _join(c["source"])
        is_current = c.get("id") == up_to_id
        if _is_prompt(c):
            msgs.append({"role": "user", "content": _strip_magic(src)})
            if not is_current:
                resp = _response_markdown(c)
                if resp: msgs.append({"role": "assistant", "content": resp})
        else:
            msgs.append({"role": "user", "content": src})
            for o in c.get("outputs", []):
                txt = _output_text(o)
                if txt: msgs.append({"role": "user", "content": f"# Output:\n{txt}"})
        if is_current: break
    return msgs

### Tests

Focused asserts on the cell→message transformation. Run with `nbdev_test`.


In [ ]:
# tiny builders so each test reads as data, not boilerplate
def _cell(cid, src, outputs=None, cell_type="code"):
    return {"id": cid, "cell_type": cell_type, "source": src, "outputs": outputs or []}
def _out_stream(text): return {"output_type": "stream", "text": text}
def _out_data(**data): return {"output_type": "execute_result", "data": data}

SYS = system_prompt

In [ ]:
# plain code cell → one user message; system prompt is first; override works
cells = [_cell("a", "x = 1"), _cell("b", "%%prompt\nhi")]
assert notebook_to_messages(cells, "b") == [
    {"role": "system", "content": SYS},
    {"role": "user",   "content": "x = 1"},
    {"role": "user",   "content": "hi"},
]
assert notebook_to_messages(cells, "b", system="S")[0] == {"role": "system", "content": "S"}

In [ ]:
# output extraction: stream, markdown, plain, and the markdown>plain preference
cells = [
    _cell("a", "print('hi')", outputs=[_out_stream(["hi\n"])]),
    _cell("b", "x",           outputs=[_out_data(**{"text/markdown": ["**bold**"]})]),
    _cell("c", "y",           outputs=[_out_data(**{"text/plain":    ["plain"]})]),
    _cell("d", "z",           outputs=[_out_data(**{"text/markdown": ["MD"],
                                                    "text/plain":    ["PL"]})]),
    _cell("e", "%%prompt\nq"),
]
got = [m["content"] for m in notebook_to_messages(cells, "e") if m["role"] == "user"]
assert got == ["print('hi')", "# Output:\nhi\n",
               "x",           "# Output:\n**bold**",
               "y",           "# Output:\nplain",
               "z",           "# Output:\nMD",
               "q"], got

In [ ]:
# documented gap: image-only outputs are silently dropped (no text representation).
# If a future change starts forwarding images, this assertion will trip — update it intentionally.
cells = [
    _cell("a", "plot()", outputs=[_out_data(**{"image/png": "BASE64..."})]),
    _cell("b", "%%prompt\nwhat is it?"),
]
msgs = notebook_to_messages(cells, "b")
assert [m["role"] for m in msgs] == ["system", "user", "user"]
assert all("BASE64" not in m["content"] for m in msgs)

In [ ]:
# prompt cells: current strips the magic and has no assistant tail;
# prior prompt with a cached markdown response yields user+assistant; without one, just user
prior_with = _cell("a", "%%prompt\nfirst",
                   outputs=[_out_data(**{"text/markdown": ["answered"]})])
prior_no   = _cell("b", "%%prompt\nsecond", outputs=[])
current    = _cell("c", "%%prompt -f\nthird")
roles = [m["role"] for m in notebook_to_messages([prior_with, prior_no, current], "c")]
assert roles == ["system", "user", "assistant", "user", "user"], roles
# current prompt's source has the magic line stripped
msgs = notebook_to_messages([current], "c")
assert msgs[-1] == {"role": "user", "content": "third"}

In [ ]:
# up_to_id stops iteration; cells beyond are not included.
# Also: source provided as a list of lines is joined identically to a string.
cells = [
    _cell("a", ["x ", "= ", "1"]),       # list source
    _cell("b", "%%prompt\nstop here"),
    _cell("c", "never_seen = True"),
]
msgs = notebook_to_messages(cells, "b")
assert [m["content"] for m in msgs[1:]] == ["x = 1", "stop here"]
assert all("never_seen" not in m["content"] for m in msgs)

Sanity check on the demo notebook:

In [ ]:
demo = json.loads(Path("99_small_demo.ipynb").read_text())["cells"]
last = demo[-1]["id"]
[(m["role"], m["content"][:60]) for m in notebook_to_messages(demo, last)]

[('system', 'You are an AI assistant embedded inside an IPython notebook '),
 ('user', '# Small demo\n> Small test subject used to understand and tes'),
 ('user', 'from nbdialog.core import *\nfrom nbdialog.providers.azure im'),
 ('user', 'write me hello world in python, like a pirate!'),
 ('assistant', '```python\nprint("Ahoy, world!")\n```'),
 ('user', 'def hello():\n    print("Hello matey!")\n\nhello()'),
 ('user', '# Output:\nHello matey!\n'),
 ('user', 'which is better?'),
 ('assistant',
  'Yours, for pirate flavor.\n\n- Mine is the classic minimal “he'),
 ('user', 'Give me top 5 news about bitcoin as of May 12 2026')]

## Providers & tools

The magic doesn't know or care which vendor answers the prompt — it only needs a callable that turns messages into text. We encode that contract as a `typing.Protocol` rather than an ABC so a provider can be any duck-typed object the user happens to have, without inheriting from anything in this package. The optional `tools` parameter lets a provider, if it supports tool calling, run a request/response loop with the model and dispatch tool invocations against Python callables.

Tools are paired (schema, function) values: an OpenAI-shaped JSON tool envelope plus the Python implementation. We deliberately keep the schema explicit and hand-written for now — generating it from type hints/docstrings is a nice future addition, not a hard requirement.

Both registries are module-level singletons because a notebook kernel *is* a single global session — there is nothing for a context manager or thread-local to scope. Users register both once at the top of their notebook:

```python
from nbdialog.providers.azure import AzureProvider
from nbdialog.tools.search import web_search
set_provider(AzureProvider())
set_tools([web_search])
```

In [ ]:
#| export
from typing import Protocol, runtime_checkable

@runtime_checkable
class Provider(Protocol):
    "A vendor-agnostic chat-completion contract: messages in, text out, plus optional tools and trace."
    def complete(self, messages: list[dict], tools: list = None, trace: "Trace | None" = None) -> str: ...

In [ ]:
#| export
from typing import NamedTuple, Callable

class Tool(NamedTuple):
    "A schema/function pair the model can call. `schema` is an OpenAI-shaped tool envelope."
    schema: dict
    fn: Callable

_provider: "Provider | None" = None
_tools: list[Tool] = []

def set_provider(p: Provider) -> None:
    "Register `p` as the provider the `%%prompt` magic will call."
    global _provider
    _provider = p

def get_provider() -> Provider:
    "Return the active provider, or raise with the fix in the message."
    if _provider is None:
        raise RuntimeError("Call nbdialog.set_provider(AzureProvider()) first.")
    return _provider

def set_tools(tools: list[Tool]) -> None:
    "Register the `Tool`s available on every `%%prompt`."
    global _tools
    _tools = list(tools)

def get_tools() -> list[Tool]:
    "Return the currently registered tools."
    return _tools

## Tracing the immediate loop

Opt-in transparency for `%%prompt`. A `Trace` records *just* the loop that produces the answer — the user's prompt, each model turn, each tool call/result, timings, and token usage. It deliberately excludes the system prompt and the notebook history that got built into `messages`; those don't change between runs of a given cell, and the interesting story is what the model did with them.

Rendering is a single collapsible disclosure block, designed to live in the cell's stored outputs so it persists across notebook saves. Because we return HTML via `_repr_html_`, the Jupyter save flow handles persistence for us — the same mechanism that already caches the markdown answer.

In [ ]:
#| export
import html, time

_TRACE_CSS = """<style>
.nbd-trace { font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px; margin: 0.5em 0; }
.nbd-trace > summary { cursor: pointer; user-select: none; padding: 4px 8px; background: rgba(127,127,127,0.10); border-radius: 4px; }
.nbd-trace .nbd-turn { padding: 6px 10px; border-left: 2px solid rgba(127,127,127,0.3); margin: 6px 0 6px 12px; }
.nbd-trace .nbd-badge { display: inline-block; padding: 1px 6px; border-radius: 3px; font-size: 10px; font-weight: 600; margin-right: 6px; vertical-align: 1px; }
.nbd-trace .nbd-badge-prompt { background: rgba(127,127,127,0.20); color: inherit; }
.nbd-trace .nbd-badge-model  { background: rgba(64,128,255,0.18);  color: #4080ff; }
.nbd-trace .nbd-badge-tool   { background: rgba(255,165,0,0.20);   color: #d67a00; }
.nbd-trace .nbd-meta { color: rgba(127,127,127,0.85); font-size: 11px; }
.nbd-trace pre { background: rgba(127,127,127,0.08); padding: 6px 8px; border-radius: 3px; white-space: pre-wrap; word-break: break-word; margin: 4px 0; font-size: 11px; }
.nbd-trace details { margin: 4px 0; }
.nbd-trace details > summary { cursor: pointer; user-select: none; }
</style>"""

class Trace:
    "Records the immediate loop (prompt, model turns, tool calls). Renders as a collapsible HTML block."
    def __init__(self):
        self.prompt: str = ""
        self.turns: list[dict] = []
        self._t0 = time.perf_counter()
        self._tokens = 0

    def set_prompt(self, text: str) -> None: self.prompt = text

    def add_model_turn(self, content, tool_calls, usage, dt):
        self.turns.append({"kind": "model", "content": content,
                           "tool_calls": list(tool_calls or []),
                           "usage": usage, "dt": dt})
        if usage: self._tokens += usage.get("total_tokens") or 0

    def add_tool_turn(self, name, args, result, dt):
        self.turns.append({"kind": "tool", "name": name, "args": args,
                           "result": result, "dt": dt})

    @property
    def total_dt(self): return time.perf_counter() - self._t0

    def _repr_html_(self): return _render_trace_html(self)

def _render_trace_html(trace: "Trace") -> str:
    "Render a `Trace` as a collapsible HTML disclosure block."
    e = html.escape
    n_model = sum(1 for t in trace.turns if t["kind"] == "model")
    n_tool  = sum(1 for t in trace.turns if t["kind"] == "tool")
    header_bits = [f"{n_model} model turn" + ("s" if n_model != 1 else ""),
                   f"{n_tool} tool call" + ("s" if n_tool != 1 else ""),
                   f"{trace.total_dt:.1f}s"]
    if trace._tokens: header_bits.append(f"{trace._tokens:,} tokens")
    summary = "trace · " + " · ".join(header_bits)

    parts = [_TRACE_CSS, f'<details class="nbd-trace"><summary>{e(summary)}</summary>']
    if trace.prompt:
        parts.append('<div class="nbd-turn">'
                     '<span class="nbd-badge nbd-badge-prompt">prompt</span>'
                     f'<pre>{e(trace.prompt)}</pre></div>')
    for i, t in enumerate(trace.turns):
        if t["kind"] == "model":
            usage = t.get("usage") or {}
            meta = f'turn {i+1} · {t["dt"]:.2f}s'
            if usage.get("total_tokens"): meta += f' · {usage["total_tokens"]:,} tokens'
            body = []
            for tc in t["tool_calls"]:
                args_json = json.dumps(tc["args"], indent=2, ensure_ascii=False)
                body.append(f'<details><summary>calls <code>{e(tc["name"])}</code></summary>'
                            f'<pre>{e(args_json)}</pre></details>')
            if t["content"]:
                body.append(f'<pre>{e(t["content"])}</pre>')
            if not body: body = ['<span class="nbd-meta">(no content; loop exiting)</span>']
            parts.append('<div class="nbd-turn">'
                         '<span class="nbd-badge nbd-badge-model">model</span>'
                         f'<span class="nbd-meta">{e(meta)}</span>'
                         + "".join(body) + '</div>')
        else:
            args_inline = ", ".join(f'{k}={json.dumps(v, ensure_ascii=False)}'
                                    for k, v in t["args"].items())
            meta = f'{t["name"]}({args_inline}) · {t["dt"]:.2f}s · {len(t["result"]):,} chars'
            parts.append('<div class="nbd-turn">'
                         '<span class="nbd-badge nbd-badge-tool">tool</span>'
                         f'<span class="nbd-meta">{e(meta)}</span>'
                         f'<details><summary>result</summary>'
                         f'<pre>{e(t["result"])}</pre></details></div>')
    parts.append('</details>')
    return "".join(parts)

## The `%%prompt` magic

Flow on every execution:

1. Parse the magic line (`-f` / `--force`).
2. Read this cell's id from `parent_header.metadata.cellId`.
3. If the on-disk copy of the cell already has rendered output and `--force` was not passed, replay that output and stop — no API call.
4. Otherwise build the message list from all cells up to and including this one, call the model, and display the response.

The cache lives in the `.ipynb` file itself: outputs are persisted by the normal Jupyter save flow, so they survive kernel restarts and Run-All-from-top, which is the whole point of this feature.

In [ ]:
#| export
def _parse_prompt_args(line):
    "Parse the magic line, e.g. `%%prompt -f --trace`."
    p = argparse.ArgumentParser(prog="%%prompt", add_help=False)
    p.add_argument("-f", "--force", action="store_true",
                   help="bypass cache and call the model even if this cell has output")
    p.add_argument("--trace", action="store_true",
                   help="render the model/tool loop alongside the final answer")
    return p.parse_args(shlex.split(line or ""))

In [ ]:
#| export
from IPython.display import HTML

def _cached_trace_html(cell):
    "Return the persisted trace HTML stored in a cell's outputs, or None."
    for o in cell.get("outputs", []):
        h = o.get("data", {}).get("text/html")
        if h:
            s = _join(h)
            if "nbd-trace" in s: return s
    return None

@register_cell_magic
def prompt(line, cell):
    "Send the notebook-so-far to the LLM and render its reply as markdown."
    args = _parse_prompt_args(line)
    ip = get_ipython()
    cell_id = ip.parent_header.get("metadata", {}).get("cellId")
    if not cell_id:
        raise RuntimeError("No cellId in parent_header — requires JupyterLab ≥ 3.5")

    nb_path = _current_notebook()
    cells = json.loads(nb_path.read_text())["cells"]
    current = next((c for c in cells if c.get("id") == cell_id), None)

    if not args.force and current:
        cached = _response_markdown(current)
        if cached:
            if args.trace:
                cached_trace = _cached_trace_html(current)
                if cached_trace: display(HTML(cached_trace))
            display(Markdown(cached))
            return

    trace = Trace() if args.trace else None
    if trace is not None: trace.set_prompt(cell)
    messages = notebook_to_messages(cells, up_to_id=cell_id)
    answer = get_provider().complete(messages, tools=get_tools() or None, trace=trace)
    if trace is not None: display(trace)
    display(Markdown(answer))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()